# Detecting inherent linearity in large CNNs

This Notebook serves to experiment with detecting inherent linearity in ResNets. These experiments will be run on smaller instances of ResNets to test functionality and provide evidence for a feasibility study as part of the graduation preparation phase. Larger experiments for the final paper(s) will be handled using Python files in this repository.

In [7]:
import torch
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from tqdm import tqdm

# --- CUDA sanity ---
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))
print(torch.version.cuda)

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print(torch.backends.cuda.matmul.allow_tf32)
print(torch.backends.cudnn.allow_tf32)

True
NVIDIA GeForce RTX 4080 SUPER
13.0
True
True


In [8]:
# --- ImageNet preprocessing ---
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])

# --- Model (ImageNet-native) ---
model = resnet18(weights=ResNet18_Weights.DEFAULT).cuda().eval()

# --- ImageNet dataset ---
# Directory structure must be:
# data/imagenet_val/
#   class1/
#   class2/
#   ...
test_dataset = ImageFolder(
    root="./data/imagenet_val",
    transform=transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    # persistent_workers=True,
    # timeout=30
)

### Verifying the model and data loader work correctly

In [9]:
def verify_model_and_data_loader(model, data_loader, device="cuda", num_batches=1):
    """
    Verify that the model and data loader work correctly by performing a forward
    pass on ImageNet-sized inputs.

    Args:
        model: ImageNet-native neural network (e.g. ResNet-18)
        data_loader: DataLoader over ImageNet val (ImageFolder)
        device: Device to run computations on
        num_batches: Number of batches to run

    Returns:
        torch.Tensor: Concatenated outputs from the first num_batches
    """

    # --- move model once ---
    model = model.to(device).eval()

    outputs = []

    with torch.no_grad():
        for inputs, _ in tqdm(data_loader, desc="Doing ImageNet forward passes", leave=False):
            # ImageNet inputs: N×3×224×224
            inputs = inputs.to(device)

            output = model(inputs)
            outputs.append(output)

            if len(outputs) >= num_batches:
                break

    return torch.cat(outputs, dim=0)

outputs = verify_model_and_data_loader(model, test_loader)
print(outputs.shape)

torch.Size([8, 1000])


"**Mean of preactivations $\bar{p}^l$** We denote the distribution of inputs $z$ to a nonlinear activation function $f(z)$ as the preactivations. For each activation function/node, we compute the mean of the preactivations, and then we compute another mean of these values per layer $l$: $\bar{p}^l= \frac{1}{M}\sum_{i=1}^M \left(\frac{1}{N}\sum_{s=1}^N z_{s,i}^l\right)$,
with $M$ the number of nodes in layer $l$ and $N$ the number of samples, and $z_{s,i}^l$ the preactivation value for sample $s$ at node $i$ at layer $l$. We compute this value through randomly selecting 500 samples of the input data instead of the whole dataset, which significantly reduces the computational cost." (Pinson et al., 2024, p. 3).

In case there is BatchNormalization applied between the convolution and the activation function, the mean of the preactivations will be approximately 0, due to the normalization. However, BN has two learned parameters per channel, namely a scaling and a shifting parameter. The shifting parameter can be used to recover the mean of the preactivations before BN. Therefore, in case of BN, we will use the shifting parameter as the mean of the preactivations. (Pinson et al., 2024)

In [10]:
def retrieve_mean_preactivations(model, data_loader, device='cuda'):
    """Compute the mean of preactivations for each ReLU layer in the model. Function does this by reading the learned shift parameter of the preceding BatchNormalization layer, as the shift will cause an inaccurate mean when computed at the ReLU layer. Returns a dictionary with layer names as keys and mean preactivation values as values.
    Args:
        model: The neural network model.
        data_loader: DataLoader for the input data.
        device: Device to run the computations on.

    Returns:
        dict: A dictionary with layer names as keys and mean preactivation values as values.
    """
    model.to(device)
    model.eval()

    previous_bn = {}
    mean_preactivations = {}
    hooks = []

    def get_hook(name):
        def hook(module, input, output):
            # If the layer is BatchNorm, save it with its shift parameter
            if isinstance(module, torch.nn.BatchNorm2d):
                shift = module.bias.mean().item()
                previous_bn[name] = shift
            # If the layer is ReLU, check if there was a preceding BatchNorm
            elif isinstance(module, torch.nn.ReLU):
                if len(previous_bn.keys()) > 0:
                    bn_name, shift = previous_bn.keys()[-1], previous_bn.popitem()[1]
                    bn_name = bn_name.split('.bn')[0]  # Get the layer name before .bn
                    assert bn_name in name, "BatchNorm layer is not part of the ReLU layer as expected."
                    mean_preactivations[name] = shift
                    previous_bn = None
                else:
                    # Compute mean preactivation directly
                    print("Warning: No preceding BatchNorm found for ReLU layer", name)
                    preactivation = input[0]
                    mean_value = preactivation.mean().item()
                    mean_preactivations[name] = mean_value




        return hook

    for name, module in tqdm(model.named_modules(), desc="Registering hooks", leave=False):
        if isinstance(module, (torch.nn.ReLU, torch.nn.BatchNorm2d)):
            hooks.append(module.register_forward_hook(get_hook(name)))
    with torch.no_grad():
        for inputs, _ in tqdm(data_loader, desc="Computing mean preactivations", leave=False):
            inputs = inputs.to(device)
            model(inputs)
            break  # Only need one batch for mean preactivations
    for hook in tqdm(hooks, desc="Removing hooks", leave=False):
        hook.remove()
    return mean_preactivations


mean_preacts = retrieve_mean_preactivations(model, test_loader)
print(mean_preacts)

UnboundLocalError: cannot access local variable 'previous_bn' where it is not associated with a value

In [6]:
# Only list non-zero mean preactivations and show neatly
# for layer, mean in mean_preacts.items():
#     if abs(mean) > 1e-6:
#         print(f"Layer: {layer}, Mean preactivation: {mean:.6f}")

# Print all model layers neatly, and remark the mean preactivations for ReLU and the two learned parameters of BatchNorm2d
for name, module in model.named_modules():
    if isinstance(module, torch.nn.ReLU):
        mean = mean_preacts.get(name, 0.0)
        print(f"Layer: {name} (ReLU), Mean preactivation: {mean:.6f}")
    elif isinstance(module, torch.nn.BatchNorm2d):
        gamma = model.state_dict()[f"{name}.weight"].mean().item()
        beta = model.state_dict()[f"{name}.bias"].mean().item()
        print(f"Layer: {name} (BatchNorm2d), gamma (scale): {gamma:.6f}, beta (shift/mean preactivation): {beta:.6f}")
    else:
        print(f"Layer: {name} ({module.__class__.__name__})")


Layer:  (ResNet)
Layer: conv1 (Conv2d)
Layer: bn1 (BatchNorm2d), gamma (scale): 0.257577, beta (shift/mean preactivation): 0.181120
Layer: relu (ReLU), Mean preactivation: 0.259286
Layer: maxpool (MaxPool2d)
Layer: layer1 (Sequential)
Layer: layer1.0 (BasicBlock)
Layer: layer1.0.conv1 (Conv2d)
Layer: layer1.0.bn1 (BatchNorm2d), gamma (scale): 0.339601, beta (shift/mean preactivation): -0.034137
Layer: layer1.0.relu (ReLU), Mean preactivation: 0.452282
Layer: layer1.0.conv2 (Conv2d)
Layer: layer1.0.bn2 (BatchNorm2d), gamma (scale): 0.333055, beta (shift/mean preactivation): 0.003463
Layer: layer1.1 (BasicBlock)
Layer: layer1.1.conv1 (Conv2d)
Layer: layer1.1.bn1 (BatchNorm2d), gamma (scale): 0.328692, beta (shift/mean preactivation): -0.083574
Layer: layer1.1.relu (ReLU), Mean preactivation: 0.515948
Layer: layer1.1.conv2 (Conv2d)
Layer: layer1.1.bn2 (BatchNorm2d), gamma (scale): 0.392430, beta (shift/mean preactivation): -0.029984
Layer: layer2 (Sequential)
Layer: layer2.0 (BasicBlock)
